# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you in loading, exploring, and analyzing the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) using the `mlcroissant` library. The dataset documents human protein data from extracellular vesicle mass spectrometry studies.

### Dataset Source
The dataset is described via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).


In [ ]:
# Ensure `mlcroissant` is installed and up-to-date
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and instantiate the dataset with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Explore record sets, their fields, and associated `@id` values.


In [ ]:
# List all record sets in the dataset (by @id and name)
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in metadata (schema may have them nested in 'distribution' items).\nTrying to infer via distribution...")
    # Sometimes, Croissant datasets define their tables/files in `distribution`,
    # each as an object with an '@id' and fields/columns listed in a schema or dictionary
    if hasattr(dataset.metadata, "distribution"):
        for dist in dataset.metadata.distribution:
            print(f"Distribution @id: {dist['@id']}")
        # We will need to access fields once we have the record set @id
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")

In [ ]:
# For this dataset, explore actual record set @ids registered
import json

# Let's print all candidate data files and their structure
distribution = getattr(dataset.metadata, 'distribution', None)

candidate_recordset_ids = []
if distribution is not None:
    for dist in distribution:
        print(f"Distribution -- @id: {dist['@id']}, Encoding: {dist.get('encodingFormat')}")
        candidate_recordset_ids.append(dist['@id'])
else:
    print("No distribution entries present.")

#### Fields available in each record set (table)
For each record/table, list fields and their `@id`s as referenced by Croissant. We'll examine the first record set in the table.


In [ ]:
# Try to get fields for the first record set (main table)
if candidate_recordset_ids:
    main_recordset_id = candidate_recordset_ids[0]
    print(f"Using main record set: {main_recordset_id}")
    # Access corresponding 'fields' (columns) if registered in the Croissant schema
    record_sets = getattr(dataset.metadata, 'record_sets', None)
    if record_sets is not None and len(record_sets)>0:
        for rs in record_sets:
            if rs['@id'] == main_recordset_id:
                print(f"Fields in {main_recordset_id}:")
                for f in rs['fields']:
                    print(f"  field @id: {f['@id']}, name: {f.get('name','N/A')}, type: {f.get('dataType','N/A')}")
    else:
        print(f"No explicit record_sets/fields section in schema.\nWill infer columns after loading records.")
else:
    print("No record set/data file @ids found.")

## 3. Data Extraction
Load data from the chosen record set into a pandas DataFrame. All entity lookups reference the Croissant `@id`.

In [ ]:
# We'll extract records from the first/main record set found above
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

dataframes = {}

# If no record sets, use candidate_recordset_ids
if not candidate_recordset_ids:
    raise ValueError("No record set or distribution data files found in schema.")

# For demonstration, extract from all provided record sets, if >1
for record_set_id in candidate_recordset_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records)==0:
        print(f"Warning: zero records found for {record_set_id}")
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns loaded: {df.columns.tolist()}")
    print(f"First 2 records:\n{df.head(2)}\n")
# Pick the main record set for downstream steps
main_recordset_id = candidate_recordset_ids[0]
main_df = dataframes[main_recordset_id]

## 4. Exploratory Data Analysis (EDA)
Basic filtering and normalization on a numeric field using Croissant column `@id`s. Outlier removal, normalization, and grouping are illustrated with available columns.

In [ ]:
# Identify a numeric field (column) by @id. List available columns to choose from.
print("Main table columns:", main_df.columns.tolist())

# For this FAIR^2 dataset, possible numeric fields could be 'MW', 'Coverage', or peptide counts.
# Let's check typical field names for mass-spec data
# For example:
#    MW = Molecular Weight, Coverage = coverage percent, LFQIntensity = normalized abundance, 'Peptides' = peptide count, etc.
# We'll pick 'MW' if present; otherwise, pick another numeric column.
possible_numeric_fields = ['MW', 'Coverage', 'Peptides', 'LFQIntensity', 'iBAQIntensity', 'UniquePeptides']
numeric_field = None
for col in possible_numeric_fields:
    if col in main_df.columns:
        numeric_field = col
        break
if numeric_field is None:
    # fallback to first float column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            break
if numeric_field is None:
    raise ValueError("No numeric field found for analysis. Unable to proceed with EDA.")
print(f"Using numeric field for EDA: {numeric_field}\n")

# Drop NA for safety
nda = main_df[numeric_field].dropna()

# Remove apparent outliers (95th percentile)
q_hi = nda.quantile(0.95)
filtered_df = main_df[main_df[numeric_field] < q_hi].copy()
print(f"Filtered records (below 95th percentile for {numeric_field}, value < {q_hi:.3g}): {len(filtered_df)} of {len(main_df)}\n")

# Normalize (z-score)
filtered_df[numeric_field + '_norm'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
filtered_df[[numeric_field, numeric_field + '_norm']].head()

#### Group (aggregate) by a categorical field
Often, protein tables have grouping by experimental condition, accession, gene name, etc.

In [ ]:
# Try grouping by a typical identifier (e.g., 'Description', 'Gene', 'Accession')
possible_group_fields = ['Gene', 'Accession', 'Description', 'GeneName', 'Sample', 'Condition']
group_field = None
for col in possible_group_fields:
    if col in main_df.columns:
        group_field = col
        break

if group_field:
    grouped = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    print(f"Top 5 groups by mean {numeric_field} (grouped by {group_field}):")
    print(grouped.head())
else:
    print("No suitable categorical field found for grouping. Skipping group-wise aggregation.")

## 5. Visualization
Visualize the distribution of the numeric field and, if available, show field relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field distribution after filtering
fig, ax = plt.subplots(1,2, figsize=(12,4))
sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True, ax=ax[0], color='steelblue')
ax[0].set_title(f'Original Distribution of {numeric_field}')
sns.histplot(filtered_df[numeric_field], bins=30, kde=True, ax=ax[1], color='forestgreen')
ax[1].set_title(f'Filtered ({numeric_field} < {q_hi:.3g})')
plt.tight_layout()
plt.show()

if group_field:
    # For a sampled subset, boxplot values by group
    top_groups = filtered_df[group_field].value_counts().index[:5]
    plot_df = filtered_df[filtered_df[group_field].isin(top_groups)]
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field, data=plot_df)
    plt.title(f'{numeric_field} by {group_field} (Top 5 Groups)')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
We explored the FAIR^2 mass spectrometry dataset via the `mlcroissant` library, loaded record sets dynamically by their `@id`, and performed simple filtering, normalization, and aggregation based on Croissant field information. You can adjust which record sets and fields to analyze by referencing their `@id` per schema documentation. Further domain-specific analyses (e.g., protein enrichment, clustering, or cross-sample comparison) can be performed on the loaded DataFrames.
